# NB5: Swarm Validation — Global Density Maps

This notebook reproduces Figure 6 and Table 5 from the paper (Section 5.2).

A global thermospheric density map is built for 18 February 2016 — a geomagnetic storm day.
Swarm satellite observations from the same hour are scaled to GRACE altitude
and overlaid on the map to assess spatial generalisation.

**Extra dependencies:** `cartopy`, `pymsis`

In [ ]:
# ============================================================
# Colab setup — run this cell FIRST. Does nothing when run locally.
# ============================================================
import os, sys, shutil

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1) Install Git LFS (required to download the parquet file from the repo)
    os.system("apt-get install -qq git-lfs")
    os.system("git lfs install")

    # 2) Clone the repo (Git LFS automatically downloads grace_workshop_small.parquet)
    if not os.path.exists("/content/ML_TND"):
        os.system("git clone -q https://github.com/lotteat11/ML_TND /content/ML_TND")

    # 3) Install Python deps
    os.system("pip install -q xgboost scikit-learn pyarrow joblib")

    # 4) Work from the workshop/ folder so all relative paths resolve
    os.chdir("/content/ML_TND/workshop")
    
    # Verify the parquet file is present
    parquet_path = "/content/ML_TND/grace_workshop_small.parquet"
    if not os.path.exists(parquet_path):
        raise FileNotFoundError(
            f"grace_workshop_small.parquet not found at {parquet_path}. "
            "Git LFS may not have downloaded it. Check your internet connection and re-run this cell."
        )
    
    print("Colab setup complete — CWD:", os.getcwd())
else:
    print("Local run — Git LFS handles the parquet file automatically via git clone.")


In [ ]:
# Install required packages — safe to run even if already installed
# cartopy may take a few minutes on a fresh environment
%pip install -q xgboost scikit-learn pandas numpy matplotlib scipy joblib pyarrow pymsis cartopy

Load libraries and import the plotting and inference functions from the project's existing forecast scripts.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "Forecast"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb

# Reuse project functions directly
from off_track import (
    build_global_feature_grid,
    interpolate_tec_to_grid,
    add_space_weather_and_msis,
    scale_swarm_hour_to_alt_many,
    plot_msis_global,
    plot_difference_global,
    plot_swarm_track_with_line2,
    FEATURE_ORDER,
    COLS_TO_SCALE,
)
from swarm_validation import collocate_points_to_grid, regression_metrics, GridSpec

from paths import GRACE_WORKSHOP as GRACE_MERGED, TEC_RAW, SWARM_MSIS
from datetime import datetime, timezone

print("Imports OK")

Set the UTC snapshot, grid resolution, colour scale limits, and model path.

In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
# UTC snapshot for the global map — must overlap with Swarm data (2015–2016)
SELECTED_TIME = datetime(2016, 2, 18, 6, 0, 0, tzinfo=timezone.utc)
# Try other snapshots: datetime(2015, 3, 17, 12, 0, 0, ...) — St Patrick's Day storm

# Grid resolution — coarser is faster, finer looks better
# For a quick run use LAT_STEP=2.0, LON_STEP=2.0
# For publication quality use LAT_STEP=0.5, LON_STEP=0.5
LAT_STEP = 1.0
LON_STEP = 1.0

# Colour scale limits for density maps [kg m⁻³]
VMIN = 1e-12
VMAX = 6e-12

# Model saved by NB3 (or use the full trained model)
MODEL_PATH  = "nb3_model_cyclic.json"    # saved at end of NB3
SCALER_PATH = "nb3_scaler_cyclic.joblib"
# ────────────────────────────────────────────────────────────────────────────

## 1. Get GRACE mean altitude for the snapshot

The global grid is built at the GRACE mean altitude for the selected hour.  
This is the altitude the model was trained on — Swarm observations will be scaled to match it.

In [ ]:
grace = pd.read_parquet(GRACE_MERGED)
grace["time"] = pd.to_datetime(grace["grace_time"], utc=True)
grace["hour"] = grace["time"].dt.floor("H")

target_hour = pd.Timestamp(SELECTED_TIME)
hour_data = grace[grace["hour"] == target_hour]

if hour_data.empty:
    # Fallback: use overall mean altitude
    alt_km = float(grace["alt_km"].mean())
    print(f"No GRACE data at {target_hour} — using mean altitude: {alt_km:.1f} km")
else:
    alt_km = float(hour_data["alt_km"].mean())
    print(f"GRACE mean altitude at {target_hour}: {alt_km:.1f} km")

## 2. Build global feature grid

Every (lat, lon) point at the fixed altitude gets LST, DOY, and lon encoding computed from the snapshot time.

In [ ]:
lat_range = np.arange(-87.5, 90.0, LAT_STEP)
lon_range = np.arange(-180.0, 180.0, LON_STEP)

grid_df = build_global_feature_grid(SELECTED_TIME, alt_km, lat_range, lon_range)
print(f"Grid: {len(lat_range)} lats × {len(lon_range)} lons = {len(grid_df):,} points")

## 3. Attach TEC

Load the IONEX TEC map for the selected epoch and interpolate to the prediction grid.

In [ ]:
df_tec = pd.read_parquet(TEC_RAW)
df_tec["epoch"] = pd.to_datetime(df_tec["epoch"], utc=True)

tec_epoch = df_tec[df_tec["epoch"] == target_hour]
print(f"TEC grid points at {target_hour}: {len(tec_epoch):,}")

if tec_epoch.empty:
    # Nearest available epoch fallback
    nearest_epoch = df_tec.iloc[(df_tec["epoch"] - target_hour).abs().argsort()[:1]]["epoch"].iloc[0]
    tec_epoch = df_tec[df_tec["epoch"] == nearest_epoch]
    print(f"Falling back to nearest TEC epoch: {nearest_epoch}  ({len(tec_epoch):,} points)")

# Interpolate TEC to prediction grid
interp_tec = interpolate_tec_to_grid(tec_epoch, lat_range, lon_range)
interp_tec["matched_tec_value"] = interp_tec["tec_value"]
interp_tec["vtec_matched_lag"]  = interp_tec["tec_value"]  # simplified lag placeholder
interp_tec["vtec_matched_lag2"] = interp_tec["tec_value"]

grid_df = grid_df.merge(interp_tec, on=["latitude", "longitude"], how="left")
print(f"TEC NaN fraction after merge: {grid_df['matched_tec_value'].isna().mean():.1%}")

## 4. Space weather + MSIS

Fetch F10.7 and Ap for the snapshot, broadcast across the grid, then run NRLMSISE-2.1 at every point.

In [ ]:
grid_df = add_space_weather_and_msis(grid_df, SELECTED_TIME)
print(f"F10.7={grid_df['f107'].iloc[0]:.1f}   Ap={grid_df['ap_m3h'].iloc[0]:.1f}")
print(f"MSIS rho range: {grid_df['msis_rho'].min():.2e} → {grid_df['msis_rho'].max():.2e} kg m⁻³")

## 5. Model prediction on the global grid

Load the trained model and apply it to every grid point.  
`rho_pred = msis_rho × exp(predicted log_ratio)`

In [ ]:
model_path = Path(MODEL_PATH)
if not model_path.exists():
    model_path = Path("..") / "xgb_model_v3.json"
    scaler_path = Path("..") / "scaler_xgboost_X_v3.joblib"
    scaler_y_path = Path("..") / "scaler_xgboost_y_v3.joblib"
    print(f"NB3 model not found — using full trained model: {model_path}")
else:
    scaler_path = Path(SCALER_PATH)
    scaler_y_path = None

model = xgb.XGBRegressor()
model.load_model(str(model_path))
scaler_X = joblib.load(str(scaler_path))
scaler_y = joblib.load(str(scaler_y_path)) if scaler_y_path and scaler_y_path.exists() else None

# Scale features
X_grid = grid_df[FEATURE_ORDER].copy()
X_grid[COLS_TO_SCALE] = scaler_X.transform(X_grid[COLS_TO_SCALE])

y_pred_scaled = model.predict(X_grid)
if scaler_y is not None:
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
else:
    y_pred = y_pred_scaled

grid_df["log_ratio_pred"] = y_pred
grid_df["rho_pred"]       = grid_df["msis_rho"] * np.exp(grid_df["log_ratio_pred"])

print(f"rho_pred range: {grid_df['rho_pred'].min():.2e} → {grid_df['rho_pred'].max():.2e} kg m⁻³")

## 6. Global density maps

Three maps matching Figure 6 in the paper: MSIS, model prediction, and their difference.  
Panels (a) and (b) in the paper show the global fields; panels (c) and (d) overlay Swarm.

Compare the MSIS and model maps — where does the model apply the largest correction?

In [ ]:
plot_msis_global(
    grid_df,
    lon_col="longitude", lat_col="latitude", value_col="msis_rho",
    title=f"MSIS density at {alt_km:.0f} km  —  {SELECTED_TIME.strftime('%Y-%m-%d %H:%M UTC')}",
    save_path="nb6_msis_global.png",
    vmin=VMIN, vmax=VMAX,
)

Plot the model's predicted density on the global grid, same colour scale as the MSIS map.

In [ ]:
plot_msis_global(
    grid_df,
    lon_col="longitude", lat_col="latitude", value_col="rho_pred",
    title=f"Model prediction at {alt_km:.0f} km  —  {SELECTED_TIME.strftime('%Y-%m-%d %H:%M UTC')}",
    save_path="nb6_pred_global.png",
    vmin=VMIN, vmax=VMAX,
)

Plot the difference between model prediction and MSIS (the correction the model applies).

In [ ]:
plot_difference_global(
    grid_df,
    lon_col="longitude", lat_col="latitude",
    pred_col="rho_pred", msis_col="msis_rho",
    title=f"Model − MSIS  at {alt_km:.0f} km  —  {SELECTED_TIME.strftime('%Y-%m-%d %H:%M UTC')}",
    save_path="nb6_diff_global.png",
)

## 7. Load Swarm and scale to GRACE altitude

Swarm flies at a different altitude than GRACE.  
We use MSIS as a transfer function: `ρ_scaled = ρ_obs × (ρ_MSIS_at_GRACE_alt / ρ_MSIS_at_Swarm_alt)`  
This puts Swarm observations on the same altitude as the model grid.

In [ ]:
swarm = pd.read_parquet(SWARM_MSIS)
swarm["time"] = pd.to_datetime(swarm["time"], utc=True)
swarm["hour"] = swarm["time"].dt.floor("H")

swarm_hour = swarm[swarm["hour"] == target_hour].copy()
print(f"Swarm samples in hour {target_hour}: {len(swarm_hour):,}")

if swarm_hour.empty:
    # Show available hours close to target
    available = swarm["hour"].unique()
    closest = sorted(available, key=lambda t: abs((t - target_hour).total_seconds()))[:5]
    print(f"No Swarm data at that hour. Nearest available hours: {[str(h) for h in closest]}")
else:
    print(f"Swarm altitude range: {swarm_hour['alt_km'].min():.0f} – {swarm_hour['alt_km'].max():.0f} km")
    print(f"GRACE target altitude: {alt_km:.1f} km")

Plot the Swarm altitude distribution and the satellite ground track for the selected hour.

In [ ]:
# Show the altitude gap before scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(swarm_hour["alt_km"], bins=40, color="C0", alpha=0.8, label="Swarm")
axes[0].axvline(alt_km, color="C3", lw=2, linestyle="--", label=f"GRACE target ({alt_km:.0f} km)")
axes[0].set_xlabel("Altitude (km)")
axes[0].set_ylabel("Count")
axes[0].set_title("Swarm vs GRACE altitude — the gap that needs scaling")
axes[0].legend()

axes[1].scatter(swarm_hour["lon"], swarm_hour["lat"], s=3, c=swarm_hour["rho_obs"],
                cmap="turbo", alpha=0.7)
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].set_title(f"Swarm ground track — {target_hour.strftime('%Y-%m-%d %H UTC')}")
plt.colorbar(axes[1].collections[0], ax=axes[1], label="ρ_obs [kg m⁻³]")

plt.tight_layout()
plt.show()

Scale each Swarm observation from Swarm altitude to GRACE altitude using the MSIS transfer factor (Equations 3–4 in the paper).

In [ ]:
# Scale all Swarm samples to GRACE altitude using MSIS transfer factor
scaled_swarm = scale_swarm_hour_to_alt_many(
    swarm_hour,
    selected_hour=target_hour,
    target_alt_km=alt_km,
    lat_col="lat", lon_col="lon", alt_col="alt_km", rho_obs_col="rho_obs",
)
print(f"Scale factor range: {scaled_swarm['scale_to_tgt'].min():.3f} → {scaled_swarm['scale_to_tgt'].max():.3f}")
print(f"rho_obs range (raw):    {scaled_swarm['rho_obs'].min():.2e} → {scaled_swarm['rho_obs'].max():.2e}")
print(f"rho_obs range (scaled): {scaled_swarm['rho_obs_scaled_to_tgt'].min():.2e} → {scaled_swarm['rho_obs_scaled_to_tgt'].max():.2e}")

## 8. Swarm track on global prediction map

This reproduces panels (c) and (d) of Figure 6 in the paper:  
the global prediction field with scaled Swarm observations overlaid on the same colour scale.

Look at where the Swarm dots land relative to the background.  
Are there regions where the scaled observations clearly disagree with the prediction?

In [ ]:
# Rename to match the function's expected column names
scaled_swarm_plot = scaled_swarm.copy()
scaled_swarm_plot["longitude"] = scaled_swarm_plot["lon"]
scaled_swarm_plot["latitude"]  = scaled_swarm_plot["lat"]

plot_swarm_track_with_line2(
    result_df=grid_df,
    swarm_df=scaled_swarm_plot,
    value_col="rho_obs_scaled_to_tgt",
    val="rho_pred",
    title=f"Swarm track (scaled) on model prediction  —  {SELECTED_TIME.strftime('%Y-%m-%d %H:%M UTC')}",
    save_path="nb6_swarm_on_pred.png",
    fixed_limits=(VMIN, VMAX),
    draw_line=True,
    verbose=False,
)

Same map as above but with MSIS as the background instead of the model prediction.

In [ ]:
# Same but with MSIS as background — easier to see where model adds value
plot_swarm_track_with_line2(
    result_df=grid_df,
    swarm_df=scaled_swarm_plot,
    value_col="rho_obs_scaled_to_tgt",
    val="msis_rho",
    title=f"Swarm track (scaled) on MSIS  —  {SELECTED_TIME.strftime('%Y-%m-%d %H:%M UTC')}",
    save_path="nb6_swarm_on_msis.png",
    fixed_limits=(VMIN, VMAX),
    draw_line=True,
    verbose=False,
)

## 9. Collocation and metrics

Each Swarm point is snapped to the nearest model grid cell (nearest-bin collocation).  
We then compute the same metrics as throughout the workshop — bias, RMSE, MAPE, R² —  
for the model and for MSIS as a baseline.

Does the model improve on MSIS when measured against an independent satellite?

In [ ]:
grid_spec = GridSpec(lat_start=-87.5, lat_step=LAT_STEP, lon_start=-180.0, lon_step=LON_STEP)

# Rename grid columns to what collocate_points_to_grid expects
grid_for_colloc = grid_df.rename(columns={"latitude": "latitude", "longitude": "longitude"}).copy()

colloc = collocate_points_to_grid(
    points_df=scaled_swarm[["lat", "lon", "rho_obs", "rho_obs_scaled_to_tgt"]].copy(),
    grid_df=grid_for_colloc[["latitude", "longitude", "rho_pred", "msis_rho"]].copy(),
    grid=grid_spec,
    lat_col_p="lat", lon_col_p="lon",
    lat_col_g="latitude", lon_col_g="longitude",
)
colloc = colloc.dropna(subset=["rho_pred", "rho_obs_scaled_to_tgt"])
print(f"Collocated pairs: {len(colloc):,}")

Compute bias, RMSE, MAPE, and R² comparing the model and MSIS against scaled Swarm observations.

In [ ]:
obs = colloc["rho_obs_scaled_to_tgt"].to_numpy()
m_pred = regression_metrics(obs, colloc["rho_pred"].to_numpy())
m_msis = regression_metrics(obs, colloc["msis_rho"].to_numpy())

metrics_df = pd.DataFrame({"Model": m_pred, "MSIS": m_msis}).T
print("\nSwarm validation metrics (density space)")
print(metrics_df[["count", "bias", "rmse", "mape", "r2", "corr", "top5"]].to_string())

Scatter Swarm observations against model predictions and against MSIS separately.

In [ ]:
# Parity: Swarm (scaled) vs model and vs MSIS
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (pred_col, label, color) in zip(axes, [
    ("rho_pred", f"Model  RMSE={m_pred['rmse']:.2e}",  "C0"),
    ("msis_rho", f"MSIS   RMSE={m_msis['rmse']:.2e}", "C1"),
]):
    x = colloc["rho_obs_scaled_to_tgt"].to_numpy()
    y = colloc[pred_col].to_numpy()
    ax.scatter(x, y, s=6, alpha=0.4, color=color, rasterized=True)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("Swarm ρ (scaled to GRACE alt) [kg m⁻³]")
    ax.set_ylabel(f"{pred_col} [kg m⁻³]")
    ax.set_title(label)

plt.suptitle("Parity: Swarm observations vs model / MSIS")
plt.tight_layout()
plt.show()

Plot the residual distributions for model and MSIS side by side.

In [ ]:
# Residual histograms
diff_pred = colloc["rho_obs_scaled_to_tgt"] - colloc["rho_pred"]
diff_msis = colloc["rho_obs_scaled_to_tgt"] - colloc["msis_rho"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(diff_msis, bins=60, alpha=0.6, color="C1", label=f"Swarm − MSIS  (bias={m_msis['bias']:.2e})")
ax.hist(diff_pred, bins=60, alpha=0.6, color="C0", label=f"Swarm − Model  (bias={m_pred['bias']:.2e})")
ax.axvline(0, color="black", lw=1, linestyle="--")
ax.set_xlabel("Residual [kg m⁻³]")
ax.set_ylabel("Count")
ax.set_title("Residual distributions: does the model reduce the bias vs MSIS?")
ax.legend()
plt.tight_layout()
plt.show()

Plot the median residual in each 10° latitude band to check for systematic geographic bias (reproduces the latitude analysis from the paper).

In [ ]:
# Median residual by 10° latitude band
bins   = np.arange(-90, 95, 10)
labels = 0.5 * (bins[:-1] + bins[1:])
colloc_plot = colloc.copy()
colloc_plot["lat_bin"] = pd.cut(colloc_plot["lat"], bins=bins, labels=labels, include_lowest=True)

lat_med = colloc_plot.groupby("lat_bin").agg(
    model=("diff_swarm_pred" if "diff_swarm_pred" in colloc_plot.columns
           else (colloc_plot["rho_obs_scaled_to_tgt"] - colloc_plot["rho_pred"]).rename("model"),
           "median"),
).reset_index() if False else (
    pd.DataFrame({
        "lat_bin": labels,
        "model":   colloc_plot.assign(d=colloc_plot["rho_obs_scaled_to_tgt"] - colloc_plot["rho_pred"]).groupby("lat_bin")["d"].median().reindex(labels).values,
        "msis":    colloc_plot.assign(d=colloc_plot["rho_obs_scaled_to_tgt"] - colloc_plot["msis_rho"]).groupby("lat_bin")["d"].median().reindex(labels).values,
    })
)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(lat_med))
w = 0.35
ax.bar(x - w/2, lat_med["msis"],  w, label="Swarm − MSIS",  color="C1", alpha=0.85)
ax.bar(x + w/2, lat_med["model"], w, label="Swarm − Model", color="C0", alpha=0.85)
ax.axhline(0, color="black", lw=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels([f"{v:.0f}°" for v in lat_med["lat_bin"]], rotation=45)
ax.set_ylabel("Median residual [kg m⁻³]")
ax.set_title("Median (Swarm − Model/MSIS) by latitude band — where does the model over/under-correct?")
ax.legend()
plt.tight_layout()
plt.show()

## Try this

1. The default epoch (18 February 2016) matches the paper's Figure 6 exactly.  
   Compare your maps to Figure 6 — do they reproduce the same density structures?
2. Change `LAT_STEP` / `LON_STEP` to `0.5` for finer resolution  
   (the paper uses 0.2° × 0.1° — computationally expensive for a workshop).
3. Look at the metrics table. Does the improvement over MSIS match Table 5 in the paper  
   (RMSE −36.2%, R² from 0.21 to 0.68)? If not, what might explain the difference?
4. Look at the latitude-band chart. The paper notes improved spatial consistency  
   particularly in regions affected by dynamic space weather. Can you see this?

---

**End of workshop.**  
The six notebooks reproduce the full pipeline from the paper:  
data exploration → feature engineering → cyclic splitting → edge-year evaluation → warm-start forecasting → global off-track validation.